# Road Collision Analysis 2025 — Complete Colab Project

**University of the West of Scotland · Advanced Data Science**

This notebook contains **all 101,525 records** of the **labelled** Department for Transport 2025 road collision dataset. The labels have already been decoded using the supplied DfT data guide (e.g. `Slight`, `Serious`, `Fatal`, `Fine no high winds`, `Single carriageway`).

### How to use
1. In Google Colab, choose **File → Upload notebook** and upload **this `.ipynb` file**.
2. Run the cells **from top to bottom**, or select **Runtime → Run all**.
3. **No CSV upload or file searching is required**. The dataset is stored inside the notebook.
4. When you reach the final section, it can start CarApp and create a temporary public website URL.

The same Decision Tree settings as the earlier project are used; the labelled text categories are one-hot encoded before modelling. **Do not replace this with the official raw CSV** unless you also adjust the label handling.  
*Source: Department for Transport (2025), Road safety open data and 2025 data guide.*

In [ ]:
#@title 1. Load the included, fully labelled dataset (no upload required)
from pathlib import Path
import base64
import pandas as pd
from IPython.display import display

# The compressed CSV is embedded inside this notebook. No user upload is needed.
DATA_B64 = (
    '...'
)
data_dir = Path('/content/data')
data_dir.mkdir(parents=True, exist_ok=True)
compressed_path = data_dir / 'road_collisions_2025_labelled.csv.xz'
compressed_path.write_bytes(base64.b64decode(DATA_B64))
del DATA_B64

df = pd.read_csv(compressed_path, compression='xz', low_memory=False)
assert df.shape == (101525, 44), f'Unexpected dimensions: {df.shape}'
assert set(df['collision_severity'].dropna().unique()) == {'Slight', 'Serious', 'Fatal'}
print('✅ Your labelled dataset is ready. No CSV upload needed!')
print(f'Records: {df.shape[0]:,} | Columns: {df.shape[1]}')
display(df[['collision_severity', 'weather_conditions', 'light_conditions',
            'road_surface_conditions', 'road_type', 'speed_limit']].head(8))

## 2. Preprocessing and data-quality checks

We examine missing values and exact duplicates. The DfT guide uses descriptive labels for categorical codes. Numeric measurements such as speed limits remain numeric; some unknown values are retained in their original categories rather than being invented.

In [ ]:
# 2. Preprocessing checks
print('Dataset size:', df.shape)
print('Exact duplicate records:', df.duplicated().sum())
missing = df.isna().sum()
print('\nColumns with missing values:')
display(missing[missing > 0].sort_values(ascending=False).to_frame('Missing records'))

print('\nRecorded severity categories:')
display(df['collision_severity'].value_counts().rename_axis('Severity').to_frame('Collisions'))

# Make an unmodified copy for any further exploration
df_clean = df.copy()

## 3. Exploratory data analysis

We compare the distributions of slight, serious and fatal *recorded injury collisions* by environmental conditions. These counts cannot establish the risk of having an accident because traffic exposure varies by weather, lighting, time, and road type.

In [ ]:
# 3A. Severity distribution
import matplotlib.pyplot as plt

severity_counts = df_clean['collision_severity'].value_counts().reindex(
    ['Slight', 'Serious', 'Fatal'], fill_value=0
)
fig, ax = plt.subplots(figsize=(8, 4.5))
severity_counts.plot(kind='bar', ax=ax)
ax.set_title('2025 recorded collisions by severity')
ax.set_xlabel('Severity')
ax.set_ylabel('Collisions')
ax.tick_params(axis='x', rotation=0)
for i, v in enumerate(severity_counts.values):
    ax.text(i, v + 600, f'{v:,}', ha='center', fontsize=10)
fig.tight_layout()
plt.show()

In [ ]:
# 3B. Serious/fatal collision proportions by weather condition
weather = pd.crosstab(df_clean['weather_conditions'], df_clean['collision_severity'])
weather['Total'] = weather.sum(axis=1)
weather['Serious/Fatal'] = weather.get('Serious', 0) + weather.get('Fatal', 0)
weather['Serious/Fatal (%)'] = (100 * weather['Serious/Fatal'] / weather['Total']).round(2)
weather = weather.sort_values('Total', ascending=False)
display(weather[['Total','Serious/Fatal','Serious/Fatal (%)']])

# Plot percentages for weather categories with >=100 collisions
chart = weather[weather['Total'] >= 100]['Serious/Fatal (%)'].sort_values()
fig, ax = plt.subplots(figsize=(11, 5))
chart.plot(kind='barh', ax=ax)
ax.set_title('Serious/fatal share among recorded collisions, by weather')
ax.set_xlabel('Serious or fatal collisions within category (%)')
fig.tight_layout()
plt.show()

In [ ]:
# 3C. Fatal collisions by lighting and serious collisions by road surface
fig, axes = plt.subplots(1, 2, figsize=(17, 5.5))
lighting_fatal = (df_clean.loc[df_clean['collision_severity'].eq('Fatal'), 'light_conditions']
                  .value_counts().sort_values(ascending=True))
lighting_fatal.plot(kind='barh', ax=axes[0])
axes[0].set_title('Fatal collisions by lighting condition')
axes[0].set_xlabel('Fatal collisions')

surface_serious = (df_clean.loc[df_clean['collision_severity'].eq('Serious'), 'road_surface_conditions']
                   .value_counts().sort_values(ascending=True))
surface_serious.plot(kind='barh', ax=axes[1])
axes[1].set_title('Serious collisions by road surface')
axes[1].set_xlabel('Serious collisions')
fig.tight_layout()
plt.show()

print('Fatal collisions by lighting:')
display(lighting_fatal.sort_values(ascending=False).rename('Collisions').to_frame())
print('Serious collisions by road surface:')
display(surface_serious.sort_values(ascending=False).rename('Collisions').to_frame())

## 4. Decision Tree classification

The model predicts **Slight (0)** versus **Serious or Fatal (1)** for *recorded collisions*. It uses six road/environmental variables, a **stratified 80/20 train/test split**, **One-Hot Encoding** for descriptive text categories, and a depth-limited Decision Tree with balanced class weights. This keeps the analysis consistent with the results in your report.

We measure accuracy, class-wise precision/recall/F1, and use a confusion matrix to show errors. The model is **not suitable for individual safety decisions**.

In [ ]:
# 4A. Prepare data and train the Decision Tree with WORD labels
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

features = [
    'speed_limit', 'road_type', 'light_conditions',
    'weather_conditions', 'road_surface_conditions', 'urban_or_rural_area'
]
categorical_features = [name for name in features if name != 'speed_limit']

X = df_clean[features].copy()
X['speed_limit'] = pd.to_numeric(X['speed_limit'], errors='raise')
y = df_clean['collision_severity'].isin(['Serious', 'Fatal']).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer([
    ('categories', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('speed', 'passthrough', ['speed_limit'])
])
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=100,
        class_weight='balanced', random_state=42
    ))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('✅ Decision Tree trained successfully')
print(f'Training records: {len(X_train):,}; Testing records: {len(X_test):,}')
print(f'Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')
print('\nClassification report:')
print(classification_report(y_test, y_pred,
                            target_names=['Slight', 'Serious/Fatal'], zero_division=0))
print('Confusion matrix [actual rows / predicted columns]:')
print(confusion_matrix(y_test, y_pred))

In [ ]:
# 4B. Confusion matrix and simplified Decision Tree diagram
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['Slight','Serious/Fatal'],
    cmap='Blues', values_format='d', ax=ax
)
ax.set_title('Decision Tree: confusion matrix')
fig.tight_layout()
plt.show()

feature_names = model.named_steps['preprocessor'].get_feature_names_out()
feature_names = [name.replace('categories__', '').replace('speed__', '') for name in feature_names]
fig, ax = plt.subplots(figsize=(24, 11))
plot_tree(model.named_steps['classifier'], feature_names=feature_names,
          class_names=['Slight', 'Serious/Fatal'],
          max_depth=3, filled=True, rounded=True, fontsize=8, ax=ax)
ax.set_title('Decision Tree: first 3 levels shown (trained max depth = 5)')
fig.tight_layout()
plt.show()

## 5. CarApp — launch your interactive website

This notebook also contains the complete CarApp Python source. These last cells are optional if you only need the analysis. If you're preparing for your presentation, run them to install Streamlit, start CarApp and obtain a **temporary** public website address.

The app automatically uses your *included labelled dataset*. You do **not** need to upload any CSV to the app, either.

In [ ]:
#@title 5A. Install Streamlit and write the CarApp file
import base64
import subprocess
import sys
from pathlib import Path

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'streamlit>=1.35', 'plotly>=5.15'])
APP_B64 = 'IiIiQ2FyQXBwIOKAlCAyMDI1IEdyZWF0IEJyaXRhaW4gcm9hZCBjb2xsaXNpb24gYW5hbHlzaXMsIGxhYmVsbGVkIGRhdGFzZXQuCgpXb3JrcyBkaXJlY3RseSB3aXRoIHRoZSBkZXNjcmlwdGl2ZSB2YWx1ZXMgaW4gdGhlIHVzZXIncyBEZXBhcnRtZW50IGZvcgpUcmFuc3BvcnQgMjAyNSBjb2xsaXNpb24gQ1NWLiBSdW46IHN0cmVhbWxpdCBydW4gQ2FyQXBwLnB5CiIiIgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdAppbXBvcnQgcGFuZGFzIGFzIHBkCmltcG9ydCBwbG90bHkuZXhwcmVzcyBhcyBweAppbXBvcnQgc3RyZWFtbGl0IGFzIHN0CmZyb20gc2tsZWFybi5jb21wb3NlIGltcG9ydCBDb2x1bW5UcmFuc2Zvcm1lcgpmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgKGFjY3VyYWN5X3Njb3JlLCBjbGFzc2lmaWNhdGlvbl9yZXBvcnQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY29uZnVzaW9uX21hdHJpeCwgQ29uZnVzaW9uTWF0cml4RGlzcGxheSkKZnJvbSBza2xlYXJuLm1vZGVsX3NlbGVjdGlvbiBpbXBvcnQgdHJhaW5fdGVzdF9zcGxpdApmcm9tIHNrbGVhcm4ucGlwZWxpbmUgaW1wb3J0IFBpcGVsaW5lCmZyb20gc2tsZWFybi5wcmVwcm9jZXNzaW5nIGltcG9ydCBPbmVIb3RFbmNvZGVyCmZyb20gc2tsZWFybi50cmVlIGltcG9ydCBEZWNpc2lvblRyZWVDbGFzc2lmaWVyLCBwbG90X3RyZWUKCnN0LnNldF9wYWdlX2NvbmZpZyhwYWdlX3RpdGxlPSdDYXJBcHAgfCAyMDI1IFJvYWQgQ29sbGlzaW9ucycsIHBhZ2VfaWNvbj0n8J+alycsIGxheW91dD0nd2lkZScpClJPT1QgPSBQYXRoKF9fZmlsZV9fKS5wYXJlbnQKREFUQSA9IFJPT1QgLyAnZGF0YScgLyAncm9hZF9jb2xsaXNpb25zXzIwMjVfbGFiZWxsZWQuY3N2Lnh6JwpGRUFUVVJFUyA9IFsnc3BlZWRfbGltaXQnLCAncm9hZF90eXBlJywgJ2xpZ2h0X2NvbmRpdGlvbnMnLAogICAgICAgICAgICAnd2VhdGhlcl9jb25kaXRpb25zJywgJ3JvYWRfc3VyZmFjZV9jb25kaXRpb25zJywgJ3VyYmFuX29yX3J1cmFsX2FyZWEnXQpDQVRFR09SSUNBTCA9IFtmIGZvciBmIGluIEZFQVRVUkVTIGlmIGYgIT0gJ3NwZWVkX2xpbWl0J10KUExPVFMgPSB7CiAgICAnV2VhdGhlciBjb25kaXRpb25zJzogJ3dlYXRoZXJfY29uZGl0aW9ucycsCiAgICAnTGlnaHRpbmcgY29uZGl0aW9ucyc6ICdsaWdodF9jb25kaXRpb25zJywKICAgICdSb2FkIHN1cmZhY2UnOiAncm9hZF9zdXJmYWNlX2NvbmRpdGlvbnMnLAogICAgJ1JvYWQgdHlwZSc6ICdyb2FkX3R5cGUnLAogICAgJ1VyYmFuIG9yIHJ1cmFsJzogJ3VyYmFuX29yX3J1cmFsX2FyZWEnLAogICAgJ1NwZWVkIGxpbWl0JzogJ3NwZWVkX2xpbWl0JywKfQpTRVZFUklUSUVTID0gWydTbGlnaHQnLCAnU2VyaW91cycsICdGYXRhbCddCgpzdC5tYXJrZG93bignJycKPHN0eWxlPgouYmxvY2stY29udGFpbmVyIHtwYWRkaW5nLXRvcDogMS4zcmVtOyBtYXgtd2lkdGg6IDEyODBweDt9CltkYXRhLXRlc3RpZD0ic3RNZXRyaWMiXSB7YmFja2dyb3VuZDogI2YzZjdmYTsgYm9yZGVyOiAxcHggc29saWQgI2RhZTVlZDsKICBwYWRkaW5nOiAxNnB4OyBib3JkZXItcmFkaXVzOiAxMnB4O30KLmhlcm8ge2JhY2tncm91bmQ6IGxpbmVhci1ncmFkaWVudCgxMjBkZWcsICMxNDJjNDgsICMxNDc1ODApOyBib3JkZXItcmFkaXVzOiAxNnB4OwogIHBhZGRpbmc6IDI2cHggMzBweDsgY29sb3I6ICNmZmY7IG1hcmdpbi1ib3R0b206IDIwcHg7fQouaGVybyBoMSB7Y29sb3I6ICNmZmY7IG1hcmdpbjogMDsgZm9udC1zaXplOiAzNHB4O30KLmhlcm8gcCB7Y29sb3I6ICNlMmYwZjE7IG1hcmdpbjogOHB4IDAgMCAwO30KPC9zdHlsZT4KPGRpdiBjbGFzcz0iaGVybyI+PGgxPvCfmpcgQ2FyQXBwIOKAlCBSb2FkIENvbGxpc2lvbiBBbmFseXNpczwvaDE+CjxwPkdyZWF0IEJyaXRhaW4sIDIwMjUgwrcgRGF0YSBleHBsb3JhdGlvbiBhbmQgRGVjaXNpb24gVHJlZSBjbGFzc2lmaWNhdGlvbjwvcD48L2Rpdj4KJycnLCB1bnNhZmVfYWxsb3dfaHRtbD1UcnVlKQoKQHN0LmNhY2hlX2RhdGEoc2hvd19zcGlubmVyPSdMb2FkaW5nIHRoZSAyMDI1IGNvbGxpc2lvbiBkYXRhc2V04oCmJykKZGVmIGxvYWRfZGF0YSgpOgogICAgcmV0dXJuIHBkLnJlYWRfY3N2KERBVEEsIGNvbXByZXNzaW9uPSd4eicsIGxvd19tZW1vcnk9RmFsc2UpCgpAc3QuY2FjaGVfcmVzb3VyY2Uoc2hvd19zcGlubmVyPSdUcmFpbmluZyBEZWNpc2lvbiBUcmVlIHVzaW5nIGxhYmVsbGVkIGNhdGVnb3JpZXPigKYnKQpkZWYgdHJhaW5fdHJlZSgpOgogICAgIyBTYW1lIHNldHRpbmdzIGFzIHRoZSBjb3Vyc2V3b3JrIG5vdGVib29rLiBFdmVyeXRoaW5nIGlzIGZpdHRlZCB1c2luZwogICAgIyB0aGUgdHJhaW5pbmcgc3BsaXQsIG5vdCB0aGUgaGVsZC1vdXQgdGVzdCByZWNvcmRzLgogICAgZGYgPSBsb2FkX2RhdGEoKQogICAgcmVxdWlyZWQgPSBGRUFUVVJFUyArIFsnY29sbGlzaW9uX3NldmVyaXR5J10KICAgIG1pc3NpbmcgPSBbY29sIGZvciBjb2wgaW4gcmVxdWlyZWQgaWYgY29sIG5vdCBpbiBkZl0KICAgIGlmIG1pc3Npbmc6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmJ01pc3NpbmcgZGF0YXNldCBjb2x1bW5zOiB7bWlzc2luZ30nKQogICAgdmFsaWQgPSBkZlsnY29sbGlzaW9uX3NldmVyaXR5J10uaXNpbihTRVZFUklUSUVTKQogICAgWCA9IGRmLmxvY1t2YWxpZCwgRkVBVFVSRVNdLmNvcHkoKQogICAgWFsnc3BlZWRfbGltaXQnXSA9IHBkLnRvX251bWVyaWMoWFsnc3BlZWRfbGltaXQnXSwgZXJyb3JzPSdyYWlzZScpCiAgICB5ID0gZGYubG9jW3ZhbGlkLCAnY29sbGlzaW9uX3NldmVyaXR5J10uaXNpbihbJ1NlcmlvdXMnLCAnRmF0YWwnXSkuYXN0eXBlKGludCkKICAgIFhfdHJhaW4sIFhfdGVzdCwgeV90cmFpbiwgeV90ZXN0ID0gdHJhaW5fdGVzdF9zcGxpdCgKICAgICAgICBYLCB5LCB0ZXN0X3NpemU9MC4yLCByYW5kb21fc3RhdGU9NDIsIHN0cmF0aWZ5PXkKICAgICkKICAgIHByZXByb2Nlc3NvciA9IENvbHVtblRyYW5zZm9ybWVyKFsKICAgICAgICAoJ2NhdGVnb3JpZXMnLCBPbmVIb3RFbmNvZGVyKGhhbmRsZV91bmtub3duPSdpZ25vcmUnKSwgQ0FURUdPUklDQUwpLAogICAgICAgICgnc3BlZWQnLCAncGFzc3Rocm91Z2gnLCBbJ3NwZWVkX2xpbWl0J10pLAogICAgXSkKICAgIG1vZGVsID0gUGlwZWxpbmUoWwogICAgICAgICgncHJlcHJvY2Vzc29yJywgcHJlcHJvY2Vzc29yKSwKICAgICAgICAoJ2NsYXNzaWZpZXInLCBEZWNpc2lvblRyZWVDbGFzc2lmaWVyKAogICAgICAgICAgICBtYXhfZGVwdGg9NSwgbWluX3NhbXBsZXNfbGVhZj0xMDAsCiAgICAgICAgICAgIGNsYXNzX3dlaWdodD0nYmFsYW5jZWQnLCByYW5kb21fc3RhdGU9NDIsCiAgICAgICAgKSksCiAgICBdKQogICAgbW9kZWwuZml0KFhfdHJhaW4sIHlfdHJhaW4pCiAgICBwcmVkID0gbW9kZWwucHJlZGljdChYX3Rlc3QpCiAgICByZXR1cm4gbW9kZWwsIHlfdGVzdCwgcHJlZCwgbGVuKFhfdHJhaW4pLCBsZW4oWF90ZXN0KQoKaWYgbm90IERBVEEuZXhpc3RzKCk6CiAgICBzdC5lcnJvcignVGhlIGRhdGFzZXQgZmlsZSBpcyBtaXNzaW5nLiBLZWVwIHRoZSBkYXRhIGZvbGRlciBhbG9uZ3NpZGUgQ2FyQXBwLnB5LicpCiAgICBzdC5jb2RlKCdkYXRhL3JvYWRfY29sbGlzaW9uc18yMDI1X2xhYmVsbGVkLmNzdi54eicpCiAgICBzdC5zdG9wKCkKCnRyeToKICAgIGRmID0gbG9hZF9kYXRhKCkKZXhjZXB0IEV4Y2VwdGlvbiBhcyBlcnJvcjoKICAgIHN0LmVycm9yKGYnQ291bGQgbm90IHJlYWQgdGhlIGluY2x1ZGVkIGRhdGFzZXQ6IHtlcnJvcn0nKQogICAgc3Quc3RvcCgpCgpzdC5zaWRlYmFyLmhlYWRlcignRXhwbG9yZSB0aGUgcmVjb3JkcycpCnN0LnNpZGViYXIuc3VjY2Vzcygn4pyFIFlvdXIgbGFiZWxsZWQgQ1NWIGlzIGFscmVhZHkgaW5jbHVkZWQuIE5vIHVwbG9hZCBuZWVkZWQuJykKc3Quc2lkZWJhci5jYXB0aW9uKGYne2xlbihkZik6LH0gcG9saWNlLXJlcG9ydGVkIGluanVyeSBjb2xsaXNpb24gcmVjb3JkcyAoMjAyNSkuJykKc3Quc2lkZWJhci5kaXZpZGVyKCkKc3Quc2lkZWJhci5tYXJrZG93bignKipTb3VyY2U6KiogRGVwYXJ0bWVudCBmb3IgVHJhbnNwb3J0LCAyMDI1IHJvYWQgc2FmZXR5IG9wZW4gZGF0YSBhbmQgaXRzIHN1cHBsaWVkIGRhdGEgZ3VpZGUuJykKCmxldmVscyA9IGRmWydjb2xsaXNpb25fc2V2ZXJpdHknXS52YWx1ZV9jb3VudHMoKQptZXRyaWNfY29scyA9IHN0LmNvbHVtbnMoNCkKZm9yIGNvbCwgdGl0bGUsIHZhbHVlIGluIHppcChtZXRyaWNfY29scywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBbJ1RvdGFsIGNvbGxpc2lvbnMnLCAnU2xpZ2h0JywgJ1NlcmlvdXMnLCAnRmF0YWwnXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBbbGVuKGRmKSwgaW50KGxldmVscy5nZXQoJ1NsaWdodCcsIDApKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgaW50KGxldmVscy5nZXQoJ1NlcmlvdXMnLCAwKSksIGludChsZXZlbHMuZ2V0KCdGYXRhbCcsIDApKV0pOgogICAgY29sLm1ldHJpYyh0aXRsZSwgZid7dmFsdWU6LH0nKQoKc3VtbWFyeSwgZXhwbG9yYXRpb24sIHF1YWxpdHksIG1vZGVsbGluZyA9IHN0LnRhYnMoWwogICAgJ/Cfk4ogT3ZlcnZpZXcnLCAn8J+UjiBDb2xsaXNpb24gQW5hbHlzaXMnLCAn8J+nuSBEYXRhIFByZXByb2Nlc3NpbmcnLCAn8J+MsyBEZWNpc2lvbiBUcmVlJwpdKQoKd2l0aCBzdW1tYXJ5OgogICAgYSwgYiA9IHN0LmNvbHVtbnMoMikKICAgIHdpdGggYToKICAgICAgICBzdC5zdWJoZWFkZXIoJ0NvbGxpc2lvbiBzZXZlcml0eScpCiAgICAgICAgY291bnRzID0gbGV2ZWxzLnJlaW5kZXgoU0VWRVJJVElFUywgZmlsbF92YWx1ZT0wKS5yZXNldF9pbmRleCgpCiAgICAgICAgY291bnRzLmNvbHVtbnMgPSBbJ1NldmVyaXR5JywgJ0NvbGxpc2lvbnMnXQogICAgICAgIGZpZyA9IHB4LmJhcihjb3VudHMsIHg9J1NldmVyaXR5JywgeT0nQ29sbGlzaW9ucycsIGNvbG9yPSdTZXZlcml0eScsCiAgICAgICAgICAgICAgICAgICAgIGNvbG9yX2Rpc2NyZXRlX21hcD17J1NsaWdodCc6ICcjMzE4QzgyJywgJ1NlcmlvdXMnOiAnI0U2QTE0QScsICdGYXRhbCc6ICcjQzk1MzVEJ30sCiAgICAgICAgICAgICAgICAgICAgIHRleHQ9J0NvbGxpc2lvbnMnKQogICAgICAgIGZpZy51cGRhdGVfdHJhY2VzKHRleHR0ZW1wbGF0ZT0nJXt0ZXh0Oix9JywgdGV4dHBvc2l0aW9uPSdvdXRzaWRlJykKICAgICAgICBmaWcudXBkYXRlX2xheW91dChzaG93bGVnZW5kPUZhbHNlLCBtYXJnaW49ZGljdCh0PTIwLCBiPTEwKSkKICAgICAgICBzdC5wbG90bHlfY2hhcnQoZmlnLCB1c2VfY29udGFpbmVyX3dpZHRoPVRydWUpCiAgICB3aXRoIGI6CiAgICAgICAgc3Quc3ViaGVhZGVyKCdTcGVlZCBsaW1pdHMnKQogICAgICAgIGNvdW50cyA9IGRmWydzcGVlZF9saW1pdCddLnZhbHVlX2NvdW50cygpLnNvcnRfaW5kZXgoKQogICAgICAgIGZpZyA9IHB4LmJhcih4PVtmJ3t4fSBtcGgnIGZvciB4IGluIGNvdW50cy5pbmRleF0sIHk9Y291bnRzLnZhbHVlcywKICAgICAgICAgICAgICAgICAgICAgbGFiZWxzPXsneCc6ICdTcGVlZCBsaW1pdCcsICd5JzogJ0NvbGxpc2lvbnMnfSkKICAgICAgICBmaWcudXBkYXRlX3RyYWNlcyhtYXJrZXJfY29sb3I9JyMyODdCOTgnKQogICAgICAgIGZpZy51cGRhdGVfbGF5b3V0KG1hcmdpbj1kaWN0KHQ9MjAsIGI9MTApKQogICAgICAgIHN0LnBsb3RseV9jaGFydChmaWcsIHVzZV9jb250YWluZXJfd2lkdGg9VHJ1ZSkKICAgIHN0LnN1YmhlYWRlcignRGF0YXNldCBwcmV2aWV3IOKAlCBkZXNjcmlwdGl2ZSBjYXRlZ29yaWVzJykKICAgIHNob3dfY29scyA9IFsnY29sbGlzaW9uX2luZGV4JywgJ2RhdGUnLCAnY29sbGlzaW9uX3NldmVyaXR5JywgJ3dlYXRoZXJfY29uZGl0aW9ucycsCiAgICAgICAgICAgICAgICAgJ2xpZ2h0X2NvbmRpdGlvbnMnLCAncm9hZF9zdXJmYWNlX2NvbmRpdGlvbnMnLCAncm9hZF90eXBlJywgJ3NwZWVkX2xpbWl0J10KICAgIHN0LmRhdGFmcmFtZShkZltbYyBmb3IgYyBpbiBzaG93X2NvbHMgaWYgYyBpbiBkZl1dLmhlYWQoMjApLAogICAgICAgICAgICAgICAgIHVzZV9jb250YWluZXJfd2lkdGg9VHJ1ZSwgaGlkZV9pbmRleD1UcnVlKQogICAgc3QuY2FwdGlvbignU3BlZWQgbGltaXRzIGFuZCBjb2xsaXNpb24gY291bnRzIHJlbWFpbiBudW1lcmljYWwgYmVjYXVzZSB0aGV5IGFyZSBtZWFzdXJlbWVudHMuJykKCndpdGggZXhwbG9yYXRpb246CiAgICBzdC5zdWJoZWFkZXIoJ0hvdyBjb25kaXRpb25zIHJlbGF0ZSB0byByZWNvcmRlZCBjb2xsaXNpb24gc2V2ZXJpdHknKQogICAgYSwgYiwgYyA9IHN0LmNvbHVtbnMoMykKICAgIHdpdGggYToKICAgICAgICBmYWN0b3JfbmFtZSA9IHN0LnNlbGVjdGJveCgnQ29uZGl0aW9uIHRvIGV4YW1pbmUnLCBsaXN0KFBMT1RTKSkKICAgIHdpdGggYjoKICAgICAgICBzZWxlY3RlZCA9IHN0LnNlbGVjdGJveCgnU2V2ZXJpdHknLCBbJ1NlcmlvdXMgb3IgRmF0YWwnLCAnRmF0YWwnLCAnU2VyaW91cycsICdTbGlnaHQnLCAnQWxsJ10pCiAgICB3aXRoIGM6CiAgICAgICAgbWV0cmljID0gc3Quc2VsZWN0Ym94KCdNZWFzdXJlJywgWydOdW1iZXIgb2YgY29sbGlzaW9ucycsICdTaGFyZSB3aXRoaW4gY29uZGl0aW9uICglKSddKQogICAgY29sX25hbWUgPSBQTE9UU1tmYWN0b3JfbmFtZV0KICAgIHNlbGVjdGVkX21hc2sgPSAoCiAgICAgICAgZGZbJ2NvbGxpc2lvbl9zZXZlcml0eSddLmlzaW4oWydTZXJpb3VzJywgJ0ZhdGFsJ10pIGlmIHNlbGVjdGVkID09ICdTZXJpb3VzIG9yIEZhdGFsJyBlbHNlCiAgICAgICAgcGQuU2VyaWVzKFRydWUsIGluZGV4PWRmLmluZGV4KSBpZiBzZWxlY3RlZCA9PSAnQWxsJyBlbHNlCiAgICAgICAgZGZbJ2NvbGxpc2lvbl9zZXZlcml0eSddLmVxKHNlbGVjdGVkKQogICAgKQogICAga2V5cyA9IGRmW2NvbF9uYW1lXS5maWxsbmEoJ05vdCByZWNvcmRlZCcpLmFzdHlwZShzdHIpCiAgICBpZiBjb2xfbmFtZSA9PSAnc3BlZWRfbGltaXQnOgogICAgICAgIGtleXMgPSBrZXlzICsgJyBtcGgnCiAgICB0b3RhbHMgPSBrZXlzLnZhbHVlX2NvdW50cygpCiAgICBtYXRjaGVkID0ga2V5c1tzZWxlY3RlZF9tYXNrXS52YWx1ZV9jb3VudHMoKQogICAgZGF0YSA9IHBkLkRhdGFGcmFtZSh7J0NvbmRpdGlvbic6IHRvdGFscy5pbmRleCwKICAgICAgICAgICAgICAgICAgICAgICAgICdBbGwgcmVjb3JkZWQgY29sbGlzaW9ucyc6IHRvdGFscy52YWx1ZXMsCiAgICAgICAgICAgICAgICAgICAgICAgICAnU2VsZWN0ZWQgY29sbGlzaW9ucyc6IG1hdGNoZWQucmVpbmRleCh0b3RhbHMuaW5kZXgsIGZpbGxfdmFsdWU9MCkudmFsdWVzfSkKICAgIGRhdGFbJ1NoYXJlIHdpdGhpbiBjb25kaXRpb24gKCUpJ10gPSAoMTAwICogZGF0YVsnU2VsZWN0ZWQgY29sbGlzaW9ucyddIC8KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZGF0YVsnQWxsIHJlY29yZGVkIGNvbGxpc2lvbnMnXSkucm91bmQoMikKICAgIGlmIHNlbGVjdGVkID09ICdBbGwnOgogICAgICAgIG1ldHJpYyA9ICdOdW1iZXIgb2YgY29sbGlzaW9ucycKICAgIGdyYXBoX2NvbCA9ICgnU2VsZWN0ZWQgY29sbGlzaW9ucycgaWYgbWV0cmljID09ICdOdW1iZXIgb2YgY29sbGlzaW9ucycKICAgICAgICAgICAgICAgICBlbHNlICdTaGFyZSB3aXRoaW4gY29uZGl0aW9uICglKScpCiAgICBkYXRhID0gZGF0YS5zb3J0X3ZhbHVlcyhncmFwaF9jb2wsIGFzY2VuZGluZz1GYWxzZSkKICAgIHN0LnBsb3RseV9jaGFydChweC5iYXIoZGF0YSwgeD0nQ29uZGl0aW9uJywgeT1ncmFwaF9jb2wsIGNvbG9yX2Rpc2NyZXRlX3NlcXVlbmNlPVsnIzIxN0Y5MSddKSwKICAgICAgICAgICAgICAgICAgICB1c2VfY29udGFpbmVyX3dpZHRoPVRydWUpCiAgICBzdC5kYXRhZnJhbWUoZGF0YSwgdXNlX2NvbnRhaW5lcl93aWR0aD1UcnVlLCBoaWRlX2luZGV4PVRydWUpCiAgICBzdC5jYXB0aW9uKCdQZXJjZW50YWdlcyBkZXNjcmliZSBzZXZlcml0eSBhbW9uZyAqcmVjb3JkZWQgaW5qdXJ5IGNvbGxpc2lvbnMqIGluIGVhY2ggY29uZGl0aW9uLiAnCiAgICAgICAgICAgICAgICdUaGV5IGRvIG5vdCBtZWFzdXJlIHRoZSBsaWtlbGlob29kIG9mIGFuIGFjY2lkZW50LCBhbmQgZG8gbm90IGVzdGFibGlzaCBjYXVzYXRpb24uJykKCndpdGggcXVhbGl0eToKICAgIHN0LnN1YmhlYWRlcignRGF0YSBxdWFsaXR5IGNoZWNrcycpCiAgICBhLCBiLCBjID0gc3QuY29sdW1ucygzKQogICAgYS5tZXRyaWMoJ1Jvd3MnLCBmJ3tkZi5zaGFwZVswXTosfScpCiAgICBiLm1ldHJpYygnQ29sdW1ucycsIGYne2RmLnNoYXBlWzFdfScpCiAgICBjLm1ldHJpYygnRXhhY3QgZHVwbGljYXRlIHJvd3MnLCBmJ3tpbnQoZGYuZHVwbGljYXRlZCgpLnN1bSgpKTosfScpCiAgICBtaXNzaW5nID0gZGYuaXNudWxsKCkuc3VtKCkuc29ydF92YWx1ZXMoYXNjZW5kaW5nPUZhbHNlKQogICAgc3QuZGF0YWZyYW1lKHBkLkRhdGFGcmFtZSh7J0NvbHVtbic6IG1pc3NpbmcuaW5kZXgsICdNaXNzaW5nIHZhbHVlcyc6IG1pc3NpbmcudmFsdWVzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgJ01pc3NpbmcgJSc6IChtaXNzaW5nLnZhbHVlcyAvIGxlbihkZikgKiAxMDApLnJvdW5kKDQpfSksCiAgICAgICAgICAgICAgICAgaGlkZV9pbmRleD1UcnVlLCB1c2VfY29udGFpbmVyX3dpZHRoPVRydWUpCiAgICBzdC5pbmZvKCdTb21lIGNvbHVtbnMgbWF5IGNvbnRhaW4gY2F0ZWdvcnkgZGVzY3JpcHRpb25zIHN1Y2ggYXMg4oCcVW5rbm93buKAnSBvciAnCiAgICAgICAgICAgICfigJxEYXRhIG1pc3Npbmcgb3Igb3V0IG9mIHJhbmdl4oCdIHJhdGhlciB0aGFuIGVtcHR5IGNlbGxzLiAnCiAgICAgICAgICAgICdBIGZldyBsb2NhbC1hdXRob3JpdHkgY29kZXMgbGFja2VkIGEgbWF0Y2hpbmcgZ3VpZGUgZW50cnkgYW5kIHdlcmUgbGVmdCB1bmNoYW5nZWQuJykKCndpdGggbW9kZWxsaW5nOgogICAgc3Quc3ViaGVhZGVyKCdEZWNpc2lvbiBUcmVlIOKAlCBTbGlnaHQgdnMgU2VyaW91cy9GYXRhbCcpCiAgICBzdC53cml0ZSgnVGhlIG1vZGVsIHVzZXMgc2l4IHJvYWQgYW5kIGVudmlyb25tZW50YWwgdmFyaWFibGVzLCB3aXRoIGFuIDgwJSB0cmFpbmluZyAnCiAgICAgICAgICAgICAnYW5kIDIwJSB0ZXN0aW5nIHNwbGl0LiBUZXh0IGNhdGVnb3JpZXMgYXJlIGNvbnZlcnRlZCB1c2luZyBPbmUtSG90IEVuY29kaW5nLicpCiAgICB0cnk6CiAgICAgICAgbW9kZWwsIHlfdGVzdCwgcHJlZCwgbl90cmFpbiwgbl90ZXN0ID0gdHJhaW5fdHJlZSgpCiAgICAgICAgZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IHJlY2FsbF9zY29yZSwgZjFfc2NvcmUKICAgICAgICBhLCBiLCBjID0gc3QuY29sdW1ucygzKQogICAgICAgIGEubWV0cmljKCdUZXN0IGFjY3VyYWN5JywgZid7YWNjdXJhY3lfc2NvcmUoeV90ZXN0LCBwcmVkKTouMiV9JykKICAgICAgICBiLm1ldHJpYygnU2VyaW91cy9GYXRhbCByZWNhbGwnLCBmJ3tyZWNhbGxfc2NvcmUoeV90ZXN0LCBwcmVkKTouMiV9JykKICAgICAgICBjLm1ldHJpYygnU2VyaW91cy9GYXRhbCBGMS1zY29yZScsIGYne2YxX3Njb3JlKHlfdGVzdCwgcHJlZCk6LjJmfScpCiAgICAgICAgc3QuY2FwdGlvbihmJ1RyYWluaW5nIHJlY29yZHM6IHtuX3RyYWluOix9IMK3IFRlc3RpbmcgcmVjb3Jkczoge25fdGVzdDosfS4gJwogICAgICAgICAgICAgICAgICAgJ01vZGVsIGRlcHRoOiA1OyBtaW5pbXVtIGxlYWYgc2l6ZTogMTAwOyBiYWxhbmNlZCBjbGFzcyB3ZWlnaHRzLicpCiAgICAgICAgbGVmdCwgcmlnaHQgPSBzdC5jb2x1bW5zKFsxLjAsIDEuMF0pCiAgICAgICAgd2l0aCBsZWZ0OgogICAgICAgICAgICBzdC5tYXJrZG93bignKipDb25mdXNpb24gbWF0cml4KionKQogICAgICAgICAgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KDUuNSwgNC4yKSkKICAgICAgICAgICAgQ29uZnVzaW9uTWF0cml4RGlzcGxheShjb25mdXNpb25fbWF0cml4KHlfdGVzdCwgcHJlZCksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZGlzcGxheV9sYWJlbHM9WydTbGlnaHQnLCAnU2VyaW91cy9GYXRhbCddKS5wbG90KGF4PWF4LCBjbWFwPSdCbHVlcycsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjb2xvcmJhcj1GYWxzZSwgdmFsdWVzX2Zvcm1hdD0nZCcpCiAgICAgICAgICAgIGZpZy50aWdodF9sYXlvdXQoKQogICAgICAgICAgICBzdC5weXBsb3QoZmlnKQogICAgICAgICAgICBwbHQuY2xvc2UoZmlnKQogICAgICAgIHdpdGggcmlnaHQ6CiAgICAgICAgICAgIHN0Lm1hcmtkb3duKCcqKkNsYXNzaWZpY2F0aW9uIHJlcG9ydCoqJykKICAgICAgICAgICAgcmVwb3J0ID0gcGQuRGF0YUZyYW1lKGNsYXNzaWZpY2F0aW9uX3JlcG9ydCgKICAgICAgICAgICAgICAgIHlfdGVzdCwgcHJlZCwgdGFyZ2V0X25hbWVzPVsnU2xpZ2h0JywgJ1NlcmlvdXMvRmF0YWwnXSwKICAgICAgICAgICAgICAgIG91dHB1dF9kaWN0PVRydWUsIHplcm9fZGl2aXNpb249MCkpLlQucm91bmQoMykKICAgICAgICAgICAgc3QuZGF0YWZyYW1lKHJlcG9ydCwgdXNlX2NvbnRhaW5lcl93aWR0aD1UcnVlKQogICAgICAgICAgICBzdC53YXJuaW5nKCdUaGUgbW9kZWwgbWlzc2VzIGFyb3VuZCBoYWxmIG9mIFNlcmlvdXMvRmF0YWwgY29sbGlzaW9ucywgc28gaXQgJwogICAgICAgICAgICAgICAgICAgICAgICdpcyB1c2VmdWwgZm9yIGRlbW9uc3RyYXRpbmcgY2xhc3NpZmljYXRpb24sIG5vdCBmb3IgcmVsaWFibGUgc2FmZXR5IGRlY2lzaW9ucy4nKQogICAgICAgIHdpdGggc3QuZXhwYW5kZXIoJ1ZpZXcgc2ltcGxpZmllZCBEZWNpc2lvbiBUcmVlIGRpYWdyYW0nKToKICAgICAgICAgICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSgxOSwgOSkpCiAgICAgICAgICAgIGZlYXR1cmVfbmFtZXMgPSBtb2RlbC5uYW1lZF9zdGVwc1sncHJlcHJvY2Vzc29yJ10uZ2V0X2ZlYXR1cmVfbmFtZXNfb3V0KCkKICAgICAgICAgICAgZmVhdHVyZV9uYW1lcyA9IFtuYW1lLnJlcGxhY2UoJ2NhdGVnb3JpZXNfXycsICcnKS5yZXBsYWNlKCdzcGVlZF9fJywgJycpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIG5hbWUgaW4gZmVhdHVyZV9uYW1lc10KICAgICAgICAgICAgcGxvdF90cmVlKG1vZGVsLm5hbWVkX3N0ZXBzWydjbGFzc2lmaWVyJ10sIGZlYXR1cmVfbmFtZXM9ZmVhdHVyZV9uYW1lcywKICAgICAgICAgICAgICAgICAgICAgIGNsYXNzX25hbWVzPVsnU2xpZ2h0JywgJ1NlcmlvdXMvRmF0YWwnXSwgbWF4X2RlcHRoPTMsCiAgICAgICAgICAgICAgICAgICAgICByb3VuZGVkPVRydWUsIGZpbGxlZD1UcnVlLCBmb250c2l6ZT03LCBheD1heCkKICAgICAgICAgICAgZmlnLnRpZ2h0X2xheW91dCgpCiAgICAgICAgICAgIHN0LnB5cGxvdChmaWcpCiAgICAgICAgICAgIHBsdC5jbG9zZShmaWcpCiAgICAgICAgICAgIHN0LmNhcHRpb24oJ1Nob3dzIG9ubHkgdGhlIGZpcnN0IHRocmVlIGxldmVsczsgdGhlIHRyYWluZWQgdHJlZSBjYW4gcmVhY2ggZGVwdGggZml2ZS4nKQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlcnJvcjoKICAgICAgICBzdC5lcnJvcihmJ1RoZSBEZWNpc2lvbiBUcmVlIGNvdWxkIG5vdCBiZSBldmFsdWF0ZWQ6IHtlcnJvcn0nKQo='
app_path = Path('/content/CarApp.py')
app_path.write_bytes(base64.b64decode(APP_B64))
print('✅ CarApp ready:', app_path)
print('✅ Labelled dataset ready:', compressed_path)

In [ ]:
# 5B. Start Streamlit once and check its health
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

health = 'http://127.0.0.1:8501/_stcore/health'
def running():
    try:
        return urllib.request.urlopen(health, timeout=3).status == 200
    except Exception:
        return False

if not running():
    log = open('/content/streamlit.log', 'w', encoding='utf-8')
    process = subprocess.Popen([
        sys.executable, '-m', 'streamlit', 'run', '/content/CarApp.py',
        '--server.address=127.0.0.1', '--server.port=8501',
        '--server.headless=true', '--browser.gatherUsageStats=false'
    ], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(30):
        if running():
            break
        time.sleep(1)

if running():
    print('✅ CarApp is running (HTTP 200).')
else:
    print('❌ CarApp could not start. Log:')
    print(Path('/content/streamlit.log').read_text(encoding='utf-8')[-5000:])

In [ ]:
# 5C. Start a temporary Cloudflare link (does not block the notebook)
import os
import re
import subprocess
import time
from pathlib import Path
import urllib.request

if not running():
    raise RuntimeError('Run step 5B first; Streamlit must be running.')

cloudflared = Path('/content/cloudflared')
if not cloudflared.exists():
    url = ('https://github.com/cloudflare/cloudflared/releases/latest/download/'
           'cloudflared-linux-amd64')
    print('Downloading Cloudflare tunnel helper...')
    urllib.request.urlretrieve(url, cloudflared)
    cloudflared.chmod(0o755)

# Reuse an existing tunnel if it is still running; otherwise start one
log_path = Path('/content/cloudflared.log')
old_proc = globals().get('tunnel_process')
if old_proc is None or old_proc.poll() is not None:
    log = open(log_path, 'w', encoding='utf-8')
    tunnel_process = subprocess.Popen(
        [str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:8501'],
        stdout=log, stderr=subprocess.STDOUT
    )

link = None
for _ in range(30):
    logs = log_path.read_text(errors='replace') if log_path.exists() else ''
    found = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', logs)
    if found:
        link = found.group(0)
        break
    time.sleep(1)

if link:
    print('✅ Open CarApp at:', link)
    print('Keep this Colab runtime running while using the website.')
else:
    print('A link has not appeared yet. Check these logs:')
    print(logs[-4000:])

### Notes for your report and presentation

- **Model accuracy:** 58.12% on 20,305 held-out collision records.
- **Serious/Fatal recall:** approximately 50%; the classifier misses about half of these collisions.
- Counts and percentages compare severity **within the police-reported injury-collision dataset**, **not the probability of a journey ending in a collision**.
- The `trycloudflare.com` address is temporary; it is not permanent deployment.
- The bundled dataset is labelled; actual measurements such as speed limits remain numeric.
- For GitHub and final ZIP submission, use the separate `CarApp.py`, `requirements.txt`, README and labelled dataset included in the companion project package. The `.ipynb` file is self-contained for Colab.

**Data reference:** Department for Transport (2025) *Road safety open data*. Available at: https://www.gov.uk/government/statistical-data-sets/road-safety-open-data